# Build `ur5_cube_clean_20ep` from the physical UR5 raw archive

This notebook preserves the original 29 raw episodes and creates a separate clean LeRobot v3 dataset. It excludes the nine episodes recorded with an unsafe action discontinuity, multi-second acquisition gaps, or multiple pick/place attempts. The exact exclusions and reasons are recorded in the output provenance report.

Run every cell from top to bottom. The selected 20 episodes are fully decoded and validated before conversion. The result is hash-verified before it becomes visible in the Drive LeRobot folder. This notebook never contacts ROS or controls the robot.

## A — Mount Drive and configure paths

Edit only `DRIVE_ROOT` or `DRIVE_RAW_ROOT` if your Drive shortcut/upload layout differs. The final Drive output is refused if it already exists unless you deliberately set `REPLACE_EXISTING_DRIVE_OUTPUT = True`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/pHYSICAL UR5")
DRIVE_RAW_ROOT = DRIVE_ROOT / "datasets/raw/episodes"
DRIVE_OUTPUT_ROOT = DRIVE_ROOT / "datasets/lerobot/ur5_cube_clean_20ep"
DRIVE_COPYING_ROOT = DRIVE_OUTPUT_ROOT.with_name(f"{DRIVE_OUTPUT_ROOT.name}.copying")

LOCAL_RAW_ROOT = Path("/content/ur5_cube_clean_20ep_raw/episodes")
LOCAL_OUTPUT_ROOT = Path("/content/ur5_cube_clean_20ep")

REPO_ID = "local/ur5_cube_clean_20ep"
TASK = "Pick up the orange cube and place it in the bowl"
EXCLUDED_EPISODES = {'episode_20260821_014409': 'multiple task attempts: four gripper transitions', 'episode_20260821_020621': 'multiple task attempts and 18 timestamp gaps; maximum gap 18.999 seconds', 'episode_20260821_021830': '11 timestamp gaps; maximum gap 3.351 seconds', 'episode_20260822_023557': 'multiple task attempts: four gripper transitions', 'episode_20260822_023942': 'multiple task attempts: four gripper transitions', 'episode_20260822_030034': 'multiple task attempts: four gripper transitions', 'episode_20260822_034925': 'four timestamp gaps; maximum gap 5.200 seconds', 'episode_20260822_035620': 'arm action discontinuity of 1.471315 radians', 'episode_20260822_035749': 'eight timestamp gaps; maximum gap 4.551 seconds'}
EXPECTED_SOURCE_EPISODES = 29
EXPECTED_EPISODES = 20
EXPECTED_FRAMES = 32022
REPLACE_EXISTING_DRIVE_OUTPUT = False

assert DRIVE_ROOT.is_dir(), f"Drive root not found: {DRIVE_ROOT}"
assert DRIVE_RAW_ROOT.is_dir(), f"Raw episodes folder not found: {DRIVE_RAW_ROOT}"
print("Raw input:", DRIVE_RAW_ROOT)
print("Final output:", DRIVE_OUTPUT_ROOT)
print("Task:", TASK)

## B — Install the pinned conversion environment

Use the same LeRobot generation as the physical dataset pipeline. The explicit PyTorch/Torchvision pair prevents the partially initialized Torchvision error previously seen in Colab. If Torch or Torchvision was already imported in this runtime, use **Runtime → Disconnect and delete runtime**, reconnect, and start again.

In [ ]:
import sys
assert "torch" not in sys.modules and "torchvision" not in sys.modules, (
    "Torch/Torchvision was already imported. Disconnect and delete the runtime, reconnect, and restart."
)

In [ ]:
%pip install -q --upgrade pip
%pip install -q --index-url https://download.pytorch.org/whl/cu126 "torch==2.7.1" "torchvision==0.22.1" "torchaudio==2.7.1"
%pip install -q "lerobot==0.4.3" "av>=12" "PyYAML>=6" "Pillow>=10"
!ffmpeg -version | head -2

In [ ]:
import importlib.metadata as md
import torch, torchvision, av

print("lerobot", md.version("lerobot"))
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("av", av.__version__)
assert md.version("lerobot") == "0.4.3"
assert torch.__version__.startswith("2.7.1")
assert torchvision.__version__.startswith("0.22.1")
assert torchvision.extension._has_ops(), "Torchvision native operators did not load"
print("Pinned environment PASS")

## C — Install the project converter inside the runtime

The notebook contains the checked-in validator/converter source, so no repository clone or extra Drive upload is required.

In [ ]:
import sys
from pathlib import Path

PACKAGE_PARENT = Path("/content/ur5_converter")
PACKAGE_DIR = PACKAGE_PARENT / "ur5_vla_dataset_recorder"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {'__init__.py': '"""UR5 VLA dataset recorder package."""\n\n', 'core.py': '"""ROS-independent validation helpers."""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Iterable, Mapping, Sequence\n\n\nUR5_JOINT_ORDER = (\n    "shoulder_pan_joint",\n    "shoulder_lift_joint",\n    "elbow_joint",\n    "wrist_1_joint",\n    "wrist_2_joint",\n    "wrist_3_joint",\n)\n\n\nclass ValidationError(ValueError):\n    """Raised when a sample cannot safely enter the dataset."""\n\n\ndef reorder_joint_positions(\n    names: Sequence[str],\n    positions: Sequence[float],\n    desired_order: Sequence[str] = UR5_JOINT_ORDER,\n) -> list[float]:\n    if len(names) != len(positions):\n        raise ValidationError("joint name and position arrays have different lengths")\n    if len(set(names)) != len(names):\n        raise ValidationError("joint state contains duplicate names")\n\n    values = dict(zip(names, positions))\n    missing = [name for name in desired_order if name not in values]\n    if missing:\n        raise ValidationError(f"missing joints: {missing}")\n\n    ordered = [float(values[name]) for name in desired_order]\n    require_finite(ordered, "joint positions")\n    return ordered\n\n\ndef reorder_command(\n    values: Sequence[float],\n    command_order: Sequence[str],\n    desired_order: Sequence[str] = UR5_JOINT_ORDER,\n) -> list[float]:\n    if len(values) != len(command_order):\n        raise ValidationError("command length does not match command_joint_order")\n    return reorder_joint_positions(command_order, values, desired_order)\n\n\ndef require_finite(values: Iterable[float], label: str) -> None:\n    if not all(math.isfinite(float(value)) for value in values):\n        raise ValidationError(f"{label} contains NaN or infinity")\n\n\ndef normalized_gripper_value(value: float, label: str) -> float:\n    result = float(value)\n    require_finite([result], label)\n    if not 0.0 <= result <= 1.0:\n        raise ValidationError(f"{label} must be in the range [0.0, 1.0]")\n    return result\n\n\ndef stream_ages(now: float, receive_times: Mapping[str, float]) -> dict[str, float]:\n    return {name: max(0.0, now - stamp) for name, stamp in receive_times.items()}\n\n\ndef stale_streams(ages: Mapping[str, float], limits: Mapping[str, float]) -> list[str]:\n    return [name for name, age in ages.items() if age > limits[name]]\n', 'raw_dataset.py': '"""ROS-independent validation and loading of staged UR5 demonstrations."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport yaml\nfrom PIL import Image\n\nfrom .core import UR5_JOINT_ORDER, reorder_command\n\nCAMERA_AGE_LIMIT_S = 0.10\nGRIPPER_AGE_LIMIT_S = 0.20\nMAX_ACTION_JUMP_RAD = 0.20\nMAX_FRAME_INTERVAL_S = 0.20\n\n\n@dataclass\nclass EpisodeValidation:\n    path: Path\n    frames: list[dict[str, Any]] = field(default_factory=list)\n    metadata: dict[str, Any] = field(default_factory=dict)\n    errors: list[str] = field(default_factory=list)\n    warnings: list[str] = field(default_factory=list)\n\n    @property\n    def ok(self) -> bool:\n        return not self.errors\n\n\ndef finalized_episode_paths(root: Path, selected: Iterable[str] | None = None) -> list[Path]:\n    wanted = set(selected or [])\n    paths = []\n    for path in sorted(root.iterdir()):\n        if not path.is_dir() or path.name.startswith(".") or path.name.endswith(".incomplete"):\n            continue\n        if wanted and path.name not in wanted:\n            continue\n        if (path / "metadata.yaml").exists() or (path / "frames.jsonl").exists():\n            paths.append(path)\n    missing = wanted - {p.name for p in paths}\n    if missing:\n        raise ValueError(f"selected episodes not found/finalized: {sorted(missing)}")\n    return paths\n\n\ndef load_rgb(path: Path) -> np.ndarray:\n    with Image.open(path) as image:\n        image.load()\n        return np.asarray(image.convert("RGB"), dtype=np.uint8)\n\n\ndef load_depth_m(path: Path, unit_m: float) -> tuple[np.ndarray, np.ndarray]:\n    with Image.open(path) as image:\n        mode = image.mode\n        raw = np.asarray(image)\n    # Pillow commonly exposes lossless 16-bit PNG mode I as int32. Verify the\n    # original mode and range before restoring its unsigned 16-bit semantics.\n    if raw.dtype == np.int32 and mode == "I" and raw.size and raw.min() >= 0 and raw.max() <= 65535:\n        raw = raw.astype(np.uint16)\n    if raw.dtype != np.uint16:\n        raise ValueError(f"depth must be uint16, got {raw.dtype}: {path}")\n    valid = (raw != 0) & (raw != np.iinfo(np.uint16).max)\n    depth = np.zeros(raw.shape, dtype=np.float32)\n    depth[valid] = raw[valid].astype(np.float32) * np.float32(unit_m)\n    return depth, valid.astype(np.uint8)\n\n\ndef _finite_vector(value: Any, length: int) -> bool:\n    return isinstance(value, list) and len(value) == length and all(\n        isinstance(x, (int, float)) and math.isfinite(float(x)) for x in value\n    )\n\n\ndef _binary(value: Any) -> bool:\n    return isinstance(value, (int, float)) and math.isfinite(float(value)) and min(\n        abs(float(value)), abs(float(value) - 1.0)\n    ) <= 1e-6\n\n\ndef validate_raw_episode(path: Path, decode_all_media: bool = True) -> EpisodeValidation:\n    result = EpisodeValidation(path=path)\n    if path.name.startswith(".") or path.name.endswith(".incomplete"):\n        result.errors.append("episode is not finalized")\n    try:\n        result.metadata = yaml.safe_load((path / "metadata.yaml").read_text())\n    except Exception as exc:\n        result.errors.append(f"metadata parse failed: {exc}")\n        return result\n    try:\n        for line_number, line in enumerate((path / "frames.jsonl").read_text().splitlines(), 1):\n            try:\n                result.frames.append(json.loads(line))\n            except Exception as exc:\n                result.errors.append(f"frames.jsonl line {line_number}: {exc}")\n    except Exception as exc:\n        result.errors.append(f"manifest read failed: {exc}")\n        return result\n\n    meta, frames = result.metadata, result.frames\n    if meta.get("outcome") != "success":\n        result.errors.append(f"outcome is {meta.get(\'outcome\')!r}, not success")\n    if tuple(meta.get("joint_order", ())) != UR5_JOINT_ORDER:\n        result.errors.append("joint_order does not match canonical UR5 order")\n    if tuple(meta.get("command_joint_order", ())) != UR5_JOINT_ORDER:\n        result.errors.append("command_joint_order does not match canonical UR5 order")\n    if meta.get("frames") != len(frames):\n        result.errors.append(f"metadata frames={meta.get(\'frames\')} but manifest has {len(frames)}")\n    indexes = [frame.get("frame_index") for frame in frames]\n    if indexes != list(range(len(frames))):\n        result.errors.append("frame indexes are not sequential from zero")\n    timestamps = np.asarray([frame.get("episode_time_s", np.nan) for frame in frames], dtype=float)\n    if not np.isfinite(timestamps).all() or (len(timestamps) > 1 and not np.all(np.diff(timestamps) > 0)):\n        result.errors.append("timestamps are non-finite or not strictly increasing")\n    elif len(timestamps) > 1:\n        intervals = np.diff(timestamps)\n        gap_indexes = np.flatnonzero(intervals > MAX_FRAME_INTERVAL_S)\n        if gap_indexes.size:\n            largest_index = int(gap_indexes[np.argmax(intervals[gap_indexes])])\n            result.errors.append(\n                f"{gap_indexes.size} frame interval(s) exceed {MAX_FRAME_INTERVAL_S:.2f}s; "\n                f"largest is {intervals[largest_index]:.6f}s between frames "\n                f"{largest_index} and {largest_index + 1}"\n            )\n\n    action_rows = []\n    gripper_commands = []\n    referenced: dict[str, list[Path]] = {name: [] for name in ("base_rgb", "wrist_rgb", "base_depth", "wrist_depth")}\n    path_keys = {\n        "base_rgb": "observation.images.base",\n        "wrist_rgb": "observation.images.wrist",\n        "base_depth": "observation.depth.base",\n        "wrist_depth": "observation.depth.wrist",\n    }\n    stale_count = 0\n    for index, frame in enumerate(frames):\n        state, action = frame.get("observation.state"), frame.get("action")\n        if not _finite_vector(state, 7):\n            result.errors.append(f"frame {index}: invalid 7D state")\n        if not _finite_vector(action, 7):\n            result.errors.append(f"frame {index}: invalid 7D action")\n        if _finite_vector(action, 7):\n            action_rows.append(action[:6])\n        values = [\n            state[6] if _finite_vector(state, 7) else None,\n            action[6] if _finite_vector(action, 7) else None,\n            frame.get("gripper.state"), frame.get("gripper.command"),\n        ]\n        if not all(_binary(value) for value in values):\n            result.errors.append(f"frame {index}: invalid binary gripper value")\n        elif values[0] != values[2] or values[1] != values[3]:\n            result.errors.append(f"frame {index}: gripper projection mismatch")\n        else:\n            gripper_commands.append(float(values[3]))\n        if frame.get("servo_status") != 0 or frame.get("valid") is not True:\n            result.errors.append(f"frame {index}: invalid frame or nonzero Servo status")\n        ages = frame.get("receive_age_s", {})\n        limits = {\n            "joint_state": CAMERA_AGE_LIMIT_S, "arm_command": CAMERA_AGE_LIMIT_S,\n            "base_image": CAMERA_AGE_LIMIT_S, "wrist_image": CAMERA_AGE_LIMIT_S,\n            "base_depth": CAMERA_AGE_LIMIT_S, "wrist_depth": CAMERA_AGE_LIMIT_S,\n            "gripper_state": GRIPPER_AGE_LIMIT_S, "gripper_command": GRIPPER_AGE_LIMIT_S,\n        }\n        if any(not math.isfinite(float(ages.get(key, math.inf))) or float(ages.get(key, math.inf)) > limit for key, limit in limits.items()):\n            stale_count += 1\n        for name, key in path_keys.items():\n            relative = frame.get(key)\n            media_path = path / relative if isinstance(relative, str) else path / "__missing__"\n            referenced[name].append(media_path)\n            if not media_path.is_file():\n                result.errors.append(f"frame {index}: missing {key}: {relative}")\n    if stale_count:\n        result.errors.append(f"{stale_count} frames contain stale observations/actions")\n    if action_rows and np.max(np.abs(np.diff(np.asarray(action_rows), axis=0)), initial=0) > MAX_ACTION_JUMP_RAD:\n        result.errors.append(f"arm action jump exceeds {MAX_ACTION_JUMP_RAD} rad/frame")\n    transitions = list(zip(gripper_commands, gripper_commands[1:]))\n    if not any(a == 1 and b == 0 for a, b in transitions):\n        result.errors.append("no gripper close transition")\n    if not any(a == 0 and b == 1 for a, b in transitions):\n        result.errors.append("no gripper open transition")\n\n    for name, paths in referenced.items():\n        directory = path / ("images" if "rgb" in name else "depth") / name.split("_")[0]\n        suffix = ".jpg" if "rgb" in name else ".png"\n        actual = set(directory.glob(f"*{suffix}")) if directory.is_dir() else set()\n        if len(actual) != len(frames) or set(paths) != actual:\n            result.errors.append(f"{name} count/reference mismatch ({len(actual)} files, {len(paths)} refs)")\n        if decode_all_media:\n            for media_path in paths:\n                if not media_path.is_file():\n                    continue\n                try:\n                    if "rgb" in name:\n                        image = load_rgb(media_path)\n                        if image.shape != (480, 640, 3):\n                            raise ValueError(f"unexpected shape {image.shape}")\n                    else:\n                        load_depth_m(media_path, float(meta.get("depth_unit_m", 0.001)))\n                except Exception as exc:\n                    result.errors.append(f"unreadable {name} {media_path.name}: {exc}")\n    if frames:\n        for index in sorted({0, len(frames) // 2, len(frames) - 1}):\n            try:\n                base = load_rgb(referenced["base_rgb"][index])\n                wrist = load_rgb(referenced["wrist_rgb"][index])\n                if hashlib.sha256(base.tobytes()).digest() == hashlib.sha256(wrist.tobytes()).digest():\n                    result.errors.append(f"frame {index}: base and wrist RGB are identical")\n            except Exception:\n                pass\n    return result\n\n\ndef canonical_state_action(frame: dict[str, Any], metadata: dict[str, Any]) -> tuple[np.ndarray, np.ndarray]:\n    """Return canonical float32 7D vectors, reordering arm components if required."""\n    state = frame["observation.state"]\n    action = frame["action"]\n    state_arm = reorder_command(state[:6], metadata["joint_order"])\n    action_arm = reorder_command(action[:6], metadata["command_joint_order"])\n    return (\n        np.asarray(state_arm + [float(state[6])], dtype=np.float32),\n        np.asarray(action_arm + [float(action[6])], dtype=np.float32),\n    )\n', 'convert_lerobot.py': '"""Convert validated staging episodes to a local LeRobotDataset v3 dataset."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .core import UR5_JOINT_ORDER\nfrom .raw_dataset import canonical_state_action, finalized_episode_paths, load_depth_m, load_rgb, validate_raw_episode\n\n\ndef feature_schema(depth: str = "none", storage: str = "video") -> dict:\n    names = [*UR5_JOINT_ORDER, "estimated_gripper_state"]\n    action_names = [*UR5_JOINT_ORDER, "gripper_command"]\n    features = {\n        "observation.images.base": {"dtype": storage, "shape": (480, 640, 3), "names": ["height", "width", "channel"]},\n        "observation.images.wrist": {"dtype": storage, "shape": (480, 640, 3), "names": ["height", "width", "channel"]},\n        "observation.state": {"dtype": "float32", "shape": (7,), "names": names},\n        "action": {"dtype": "float32", "shape": (7,), "names": action_names},\n        "source_episode_time_s": {"dtype": "float64", "shape": (1,), "names": ["seconds"]},\n    }\n    if depth == "meters":\n        for camera in ("base", "wrist"):\n            features[f"observation.depth.{camera}"] = {"dtype": "float32", "shape": (480, 640), "names": ["height", "width"]}\n            features[f"observation.depth_valid.{camera}"] = {"dtype": "uint8", "shape": (480, 640), "names": ["height", "width"]}\n    return features\n\n\ndef prepare_output(path: Path, overwrite: bool) -> None:\n    if path.exists():\n        if not overwrite:\n            raise FileExistsError(f"output exists; pass --overwrite to replace it: {path}")\n        if path.resolve() in (Path.home().resolve(), Path("/").resolve()):\n            raise ValueError("refusing unsafe output path")\n        shutil.rmtree(path)\n\n\ndef convert(input_root: Path, output_root: Path, repo_id: str, task: str | None = None,\n            episodes: list[str] | None = None, overwrite: bool = False, depth: str = "none",\n            storage: str = "video") -> tuple[int, int]:\n    from lerobot.datasets.lerobot_dataset import LeRobotDataset\n\n    paths = finalized_episode_paths(input_root, episodes)\n    if not paths:\n        raise ValueError("no finalized episodes selected")\n    validations = []\n    print(f"[preflight] validating {len(paths)} raw episodes with full media decoding", flush=True)\n    for index, path in enumerate(paths, 1):\n        print(f"[preflight {index}/{len(paths)}] validating {path.name} ...", flush=True)\n        validation = validate_raw_episode(path, decode_all_media=True)\n        validations.append(validation)\n        verdict = "PASS" if validation.ok else "FAIL"\n        print(\n            f"[preflight {index}/{len(paths)}] {verdict} {path.name}: "\n            f"{len(validation.frames)} frames",\n            flush=True,\n        )\n    failures = [item for item in validations if not item.ok]\n    if failures:\n        detail = "\\n".join(f"{item.path.name}: {\'; \'.join(item.errors[:10])}" for item in failures)\n        raise ValueError(f"raw validation failed; nothing converted:\\n{detail}")\n    print("[preflight] PASS; creating LeRobot output", flush=True)\n    prepare_output(output_root, overwrite)\n    dataset = LeRobotDataset.create(repo_id=repo_id, fps=20, features=feature_schema(depth, storage),\n                                    root=output_root, robot_type="ur5", use_videos=storage == "video",\n                                    tolerance_s=0.01)\n    total = 0\n    for episode_index, validation in enumerate(validations, 1):\n        metadata = validation.metadata\n        episode_task = task or metadata.get("task")\n        frame_count = len(validation.frames)\n        print(\n            f"[convert {episode_index}/{len(validations)}] writing {validation.path.name}: "\n            f"{frame_count} frames",\n            flush=True,\n        )\n        for frame_index, raw in enumerate(validation.frames, 1):\n            state, action = canonical_state_action(raw, metadata)\n            frame = {\n                "observation.images.base": load_rgb(validation.path / raw["observation.images.base"]),\n                "observation.images.wrist": load_rgb(validation.path / raw["observation.images.wrist"]),\n                "observation.state": state,\n                "action": action,\n                # LeRobot 0.4.3 generates its built-in regular 20 Hz timestamp. The\n                # exact original episode clock is retained as a separate feature.\n                "source_episode_time_s": np.asarray([raw["episode_time_s"]], dtype=np.float64),\n                "task": episode_task,\n            }\n            if depth == "meters":\n                for camera in ("base", "wrist"):\n                    values, valid = load_depth_m(validation.path / raw[f"observation.depth.{camera}"], float(metadata["depth_unit_m"]))\n                    frame[f"observation.depth.{camera}"] = values\n                    frame[f"observation.depth_valid.{camera}"] = valid\n            dataset.add_frame(frame)\n            total += 1\n            if frame_index % 500 == 0 or frame_index == frame_count:\n                print(\n                    f"[convert {episode_index}/{len(validations)}] "\n                    f"{frame_index}/{frame_count} frames staged",\n                    flush=True,\n                )\n        print(\n            f"[convert {episode_index}/{len(validations)}] encoding base/wrist videos ...",\n            flush=True,\n        )\n        dataset.save_episode(parallel_encoding=False)\n        print(f"[convert {episode_index}/{len(validations)}] episode saved", flush=True)\n    print("[finalize] computing metadata/statistics and cleaning temporary assets ...", flush=True)\n    dataset.finalize()\n    conversion_meta = {\n        "source_episode_names": [item.path.name for item in validations],\n        "joint_order": list(UR5_JOINT_ORDER), "joint_unit": "radian",\n        "state_semantics": "six measured joint positions plus estimated last-successful pneumatic command",\n        "action_semantics": "six absolute joint-position targets plus absolute binary gripper command",\n        "gripper_closed_value": 0.0, "gripper_open_value": 1.0,\n        "gripper_is_measured": False, "depth_mode": depth,\n        "depth_note": "Raw depth remains in source episodes. meters mode masks raw 0 and 65535 to zero and stores validity masks.",\n        "image_storage": storage, "lerobot_version_target": "0.4.3 / dataset codebase v3.0",\n    }\n    (output_root / "meta" / "ur5_vla_conversion.json").write_text(json.dumps(conversion_meta, indent=2) + "\\n")\n    print(f"[finalize] complete: {len(validations)} episodes / {total} frames", flush=True)\n    return len(validations), total\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", type=Path, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--repo-id", required=True)\n    parser.add_argument("--task")\n    parser.add_argument("--episodes", nargs="*", help="episode directory names; default: all finalized")\n    parser.add_argument("--overwrite", action="store_true")\n    parser.add_argument("--depth", choices=("none", "meters"), default="none")\n    parser.add_argument("--storage", choices=("video", "image"), default="video")\n    args = parser.parse_args()\n    count, frames = convert(**vars(args))\n    print(f"Converted {count} episodes / {frames} frames to {args.output_root}")\n\n\nif __name__ == "__main__":\n    main()\n', 'validate_lerobot.py': '"""Validate a converted dataset by loading it through the official LeRobot API."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef validate_dataset(root: Path, repo_id: str, report: Path, expected_episodes: int | None = None,\n                     expected_frames: int | None = None, task: str | None = None,\n                     video_backend: str = "pyav") -> list[str]:\n    from lerobot.datasets.lerobot_dataset import LeRobotDataset\n\n    errors, warnings = [], []\n    dataset = LeRobotDataset(repo_id=repo_id, root=root, video_backend=video_backend)\n    features = dataset.meta.features\n    required = {"observation.images.base", "observation.images.wrist", "observation.state", "action"}\n    if not required.issubset(features):\n        errors.append(f"missing features: {sorted(required - set(features))}")\n    if tuple(features.get("observation.state", {}).get("shape", ())) != (7,):\n        errors.append("observation.state feature is not shape (7,)")\n    if tuple(features.get("action", {}).get("shape", ())) != (7,):\n        errors.append("action feature is not shape (7,)")\n    if expected_episodes is not None and dataset.meta.total_episodes != expected_episodes:\n        errors.append(f"expected {expected_episodes} episodes, found {dataset.meta.total_episodes}")\n    if expected_frames is not None and dataset.meta.total_frames != expected_frames:\n        errors.append(f"expected {expected_frames} frames, found {dataset.meta.total_frames}")\n    tasks = list(dataset.meta.tasks.index)\n    if task is not None and task not in tasks:\n        errors.append(f"task text missing: {task!r}")\n    metadata_path = root / "meta" / "ur5_vla_conversion.json"\n    conversion_meta = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}\n    if conversion_meta.get("action_semantics") != "six absolute joint-position targets plus absolute binary gripper command":\n        errors.append("UR5 action semantics metadata missing or incorrect")\n\n    gripper_by_episode: dict[int, list[float]] = {}\n    samples = set()\n    episodes = dataset.meta.episodes\n    for ep in range(dataset.meta.total_episodes):\n        start, stop = int(episodes[ep]["dataset_from_index"]), int(episodes[ep]["dataset_to_index"])\n        samples.update({start, (start + stop - 1) // 2, stop - 1})\n    # Scan scalar/vector parquet data for every row without needlessly decoding\n    # both videos 20k times. Visual samples below still use official __getitem__.\n    for index in range(len(dataset)):\n        sample = dataset.hf_dataset[index]\n        state = np.asarray(sample["observation.state"])\n        action = np.asarray(sample["action"])\n        if state.shape != (7,) or action.shape != (7,) or not np.isfinite(state).all() or not np.isfinite(action).all():\n            errors.append(f"frame {index}: invalid state/action shape or values")\n            continue\n        if min(abs(float(action[6])), abs(float(action[6]) - 1.0)) > 1e-5:\n            errors.append(f"frame {index}: non-binary gripper action {action[6]}")\n        ep = int(sample["episode_index"])\n        gripper_by_episode.setdefault(ep, []).append(float(action[6]))\n    for index in sorted(samples):\n        sample = dataset[index]\n        for key in ("observation.images.base", "observation.images.wrist"):\n            image = np.asarray(sample[key])\n            if image.shape not in ((3, 480, 640), (480, 640, 3)):\n                errors.append(f"sample {index}: {key} has shape {image.shape}")\n    for ep, values in gripper_by_episode.items():\n        changes = list(zip(values, values[1:]))\n        if not any(a > .5 and b < .5 for a, b in changes) or not any(a < .5 and b > .5 for a, b in changes):\n            errors.append(f"episode {ep}: missing close/open transition")\n\n    lines = ["# Converted LeRobot Dataset Validation", "", f"**Verdict: {\'FAIL\' if errors else \'PASS\'}**", "",\n             f"- Root: `{root}`", f"- Repo ID: `{repo_id}`", f"- Episodes: {dataset.meta.total_episodes}",\n             f"- Frames: {dataset.meta.total_frames}", f"- FPS: {dataset.meta.fps}",\n             f"- Features: `{\', \'.join(features)}`", f"- Tasks: `{tasks}`", f"- Sampled frames: {len(samples)}", "",\n             "## Critical errors", ""]\n    lines.extend([f"- {value}" for value in errors] or ["None."])\n    lines += ["", "## Warnings", ""]\n    lines.extend([f"- {value}" for value in warnings] or ["None."])\n    report.parent.mkdir(parents=True, exist_ok=True)\n    report.write_text("\\n".join(lines) + "\\n")\n    return errors\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--repo-id", required=True)\n    parser.add_argument("--report", type=Path, required=True)\n    parser.add_argument("--expected-episodes", type=int)\n    parser.add_argument("--expected-frames", type=int)\n    parser.add_argument("--task")\n    parser.add_argument("--video-backend", choices=("pyav", "torchcodec"), default="pyav")\n    args = parser.parse_args()\n    errors = validate_dataset(**vars(args))\n    if errors:\n        print("Validation failed:\\n- " + "\\n- ".join(errors))\n        raise SystemExit(1)\n    print(f"PASS: {args.report}")\n\n\nif __name__ == "__main__":\n    main()\n'}
for filename, source in SOURCE_FILES.items():
    (PACKAGE_DIR / filename).write_text(source)

if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

from ur5_vla_dataset_recorder.convert_lerobot import convert
from ur5_vla_dataset_recorder.raw_dataset import finalized_episode_paths
from ur5_vla_dataset_recorder.validate_lerobot import validate_dataset

print("Embedded converter installed:", PACKAGE_DIR)

## D — Inspect Drive input and local disk

The Drive folder must contain exactly 29 finalized episodes before exclusion. The known-invalid episode is retained in Drive but is never copied into the selected conversion set.

In [ ]:
import shutil

drive_episodes = finalized_episode_paths(DRIVE_RAW_ROOT)
drive_names = [path.name for path in drive_episodes]
assert len(drive_episodes) == EXPECTED_SOURCE_EPISODES, (
    f"Expected {EXPECTED_SOURCE_EPISODES} finalized Drive episodes, found {len(drive_episodes)}"
)
missing_exclusions = set(EXCLUDED_EPISODES) - set(drive_names)
assert not missing_exclusions, f"Excluded episodes missing from source: {sorted(missing_exclusions)}"

selected_names = [name for name in drive_names if name not in EXCLUDED_EPISODES]
assert len(selected_names) == EXPECTED_EPISODES, (len(selected_names), EXPECTED_EPISODES)

disk = shutil.disk_usage("/content")
print(f"Drive finalized episodes: {len(drive_episodes)}")
print(f"Selected episodes: {len(selected_names)}")
print("Excluded episodes and reasons:")
for name, reason in EXCLUDED_EPISODES.items():
    print(f"  - {name}: {reason}")
print(f"Local free space: {disk.free / 2**30:.1f} GiB")
assert disk.free >= 35 * 2**30, "At least 35 GiB of free /content storage is required"

## E — Copy raw episodes from Drive to `/content`

This is a local working copy. The original Drive upload is never modified. `rsync` displays transfer progress.

In [ ]:
import shutil, subprocess

def reset_local_directory(path: Path):
    resolved = path.resolve()
    assert resolved != Path("/content") and Path("/content") in resolved.parents, resolved
    if resolved.exists():
        shutil.rmtree(resolved)
    resolved.mkdir(parents=True, exist_ok=True)

reset_local_directory(LOCAL_RAW_ROOT)
subprocess.run(
    ["rsync", "-a", "--info=progress2", f"{DRIVE_RAW_ROOT}/", f"{LOCAL_RAW_ROOT}/"],
    check=True,
)

local_episodes = finalized_episode_paths(LOCAL_RAW_ROOT)
assert [path.name for path in local_episodes] == drive_names, "Local raw copy episode list differs from Drive"
print("Raw copy complete:", LOCAL_RAW_ROOT)

## F — Full raw validation and conversion

This cell is intentionally verbose. First it prints one `preflight` result per episode while decoding every RGB/depth file. After all 20 pass, it prints every 500 staged frames and each base/wrist AV1 encoding stage. Do not close the browser or interrupt the runtime.

The output is RGB-only for SmolVLA/LeRobot policies. Lossless raw depth remains preserved in the Drive raw upload.

In [ ]:
if LOCAL_OUTPUT_ROOT.exists():
    resolved = LOCAL_OUTPUT_ROOT.resolve()
    assert resolved != Path("/content") and Path("/content") in resolved.parents
    shutil.rmtree(resolved)

count, frames = convert(
    input_root=LOCAL_RAW_ROOT,
    output_root=LOCAL_OUTPUT_ROOT,
    repo_id=REPO_ID,
    task=TASK,
    episodes=selected_names,
    overwrite=False,
    depth="none",
    storage="video",
)

assert count == EXPECTED_EPISODES, (count, EXPECTED_EPISODES)
assert frames == EXPECTED_FRAMES, (frames, EXPECTED_FRAMES)
print(f"CONVERSION PASS: {count} episodes / {frames} frames")

## G — Validate the converted LeRobot dataset

This scans every state/action row and decodes the beginning, middle, and end views from every episode through the official LeRobot API.

In [ ]:
LOCAL_VALIDATION_REPORT = LOCAL_OUTPUT_ROOT / "validation_report.md"
errors = validate_dataset(
    root=LOCAL_OUTPUT_ROOT,
    repo_id=REPO_ID,
    report=LOCAL_VALIDATION_REPORT,
    expected_episodes=EXPECTED_EPISODES,
    expected_frames=EXPECTED_FRAMES,
    task=TASK,
    video_backend="pyav",
)
assert not errors, "Converted validation failed:\n- " + "\n- ".join(errors)
print(LOCAL_VALIDATION_REPORT.read_text())

## H — Record provenance and copy the completed dataset to Drive

The notebook copies into `ur5_cube_clean_20ep.copying`, hashes every file locally and on Drive, and renames the directory only after exact verification. A pre-existing final output is refused by default.

In [ ]:
import datetime, hashlib, json, os, shutil, subprocess

conversion_report = {
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "repo_id": REPO_ID,
    "task": TASK,
    "episodes": EXPECTED_EPISODES,
    "frames": EXPECTED_FRAMES,
    "fps": 20,
    "source_drive_root": str(DRIVE_RAW_ROOT),
    "excluded_episodes": EXCLUDED_EPISODES,
    "selection_manifest": "config/ur5_cube_clean_20ep.json",
    "selected_source_episodes": selected_names,
    "depth_mode": "none; raw depth remains in source upload",
    "storage": "video",
    "lerobot": md.version("lerobot"),
}
(LOCAL_OUTPUT_ROOT / "conversion_run_report.json").write_text(json.dumps(conversion_report, indent=2) + "\n")

def safe_drive_remove(path: Path):
    resolved = path.resolve()
    assert resolved.name in {"ur5_cube_clean_20ep", "ur5_cube_clean_20ep.copying"}
    assert "datasets/lerobot" in str(resolved)
    if resolved.exists():
        shutil.rmtree(resolved)

if DRIVE_OUTPUT_ROOT.exists() or DRIVE_COPYING_ROOT.exists():
    assert REPLACE_EXISTING_DRIVE_OUTPUT, (
        "Drive output or .copying folder exists. Inspect it first; set "
        "REPLACE_EXISTING_DRIVE_OUTPUT=True only to replace this exact dataset."
    )
    safe_drive_remove(DRIVE_OUTPUT_ROOT)
    safe_drive_remove(DRIVE_COPYING_ROOT)

DRIVE_COPYING_ROOT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["rsync", "-a", "--info=progress2", f"{LOCAL_OUTPUT_ROOT}/", f"{DRIVE_COPYING_ROOT}/"],
    check=True,
)

def file_hashes(root: Path):
    result = {}
    for path in sorted(item for item in root.rglob("*") if item.is_file()):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            while chunk := stream.read(8 * 1024 * 1024):
                digest.update(chunk)
        result[str(path.relative_to(root))] = {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}
    return result

local_hashes = file_hashes(LOCAL_OUTPUT_ROOT)
drive_hashes = file_hashes(DRIVE_COPYING_ROOT)
assert drive_hashes == local_hashes, "Drive copy hash verification failed"

DRIVE_COPYING_ROOT.rename(DRIVE_OUTPUT_ROOT)
(DRIVE_OUTPUT_ROOT / "drive_copy_manifest.json").write_text(json.dumps(local_hashes, indent=2) + "\n")
print("DRIVE COPY PASS:", DRIVE_OUTPUT_ROOT)
print("Files verified:", len(local_hashes))

## I — Final Drive verification

Do not train until this cell passes. The Drive output is loaded directly only for final verification; training should later copy it back to `/content` first.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

drive_dataset = LeRobotDataset(REPO_ID, root=DRIVE_OUTPUT_ROOT, video_backend="pyav")
assert drive_dataset.meta.total_episodes == EXPECTED_EPISODES
assert drive_dataset.meta.total_frames == EXPECTED_FRAMES
assert TASK in list(drive_dataset.meta.tasks.index)
assert (DRIVE_OUTPUT_ROOT / "validation_report.md").is_file()
assert (DRIVE_OUTPUT_ROOT / "conversion_run_report.json").is_file()
assert (DRIVE_OUTPUT_ROOT / "drive_copy_manifest.json").is_file()

print("FINAL PASS")
print("Drive dataset:", DRIVE_OUTPUT_ROOT)
print("Repository ID:", REPO_ID)
print("Episodes / frames:", EXPECTED_EPISODES, EXPECTED_FRAMES)
print("Task:", TASK)